In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = '/kaggle/input/competitions/datathon-2026-round-1/'
SALES = DATA_DIR + 'sales.csv'
ORDERS = DATA_DIR + 'orders.csv'

# Question 1:
Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [3]:
df_orders = pd.read_csv(ORDERS)
df_orders.head(10)

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign
5,7,2012-07-06,57820,2886,delivered,credit_card,tablet,organic_search
6,8,2012-07-06,57818,2886,delivered,credit_card,mobile,email_campaign
7,9,2012-07-06,49102,5262,delivered,apple_pay,tablet,paid_search
8,10,2012-07-06,49101,5262,delivered,paypal,desktop,organic_search
9,13,2012-07-06,40638,7456,delivered,credit_card,tablet,social_media


In [4]:
df_orders_groupid = df_orders.groupby(['customer_id']).size().reset_index(name='count')
df_orders_groupid

,customer_id,count
0,1,6
1,2,4
2,3,3
3,4,1
4,5,5
...,...,...
90241,157554,1
90242,157555,2
90243,157557,1
90244,157561,22


In [5]:
df_orders_groupid = df_orders_groupid[df_orders_groupid['count'] != 1]
df_orders_groupid

,customer_id,count
0,1,6
1,2,4
2,3,3
4,5,5
5,6,10
...,...,...
90234,157530,2
90238,157543,3
90242,157555,2
90244,157561,22


In [6]:
df_orders_new = df_orders[df_orders['customer_id'].isin(df_orders_groupid['customer_id'])]
df_orders_new.sort_values(['customer_id', 'order_date'], ascending = True, inplace=True)
df_orders_new

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
4023,5280,2012-07-25,1,15201,delivered,cod,desktop,paid_search
143252,184922,2014-05-31,1,15201,returned,credit_card,mobile,referral
238890,308113,2015-07-31,1,15201,delivered,cod,mobile,paid_search
374571,483190,2017-04-23,1,15201,delivered,cod,mobile,paid_search
544446,702081,2020-02-24,1,15201,delivered,credit_card,mobile,organic_search
...,...,...,...,...,...,...,...,...
549990,709267,2020-04-11,157563,59937,delivered,credit_card,mobile,email_campaign
557438,718824,2020-05-29,157563,59937,delivered,credit_card,mobile,social_media
559725,721751,2020-06-22,157563,59937,delivered,paypal,tablet,organic_search
591841,763157,2021-05-27,157563,59937,delivered,credit_card,mobile,social_media


In [7]:
print(df_orders_new['customer_id'].unique())
df_orders_new['order_date'] = pd.to_datetime(df_orders_new['order_date'])
df_orders_new

[     1      2      3 ... 157555 157561 157563]


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
4023,5280,2012-07-25,1,15201,delivered,cod,desktop,paid_search
143252,184922,2014-05-31,1,15201,returned,credit_card,mobile,referral
238890,308113,2015-07-31,1,15201,delivered,cod,mobile,paid_search
374571,483190,2017-04-23,1,15201,delivered,cod,mobile,paid_search
544446,702081,2020-02-24,1,15201,delivered,credit_card,mobile,organic_search
...,...,...,...,...,...,...,...,...
549990,709267,2020-04-11,157563,59937,delivered,credit_card,mobile,email_campaign
557438,718824,2020-05-29,157563,59937,delivered,credit_card,mobile,social_media
559725,721751,2020-06-22,157563,59937,delivered,paypal,tablet,organic_search
591841,763157,2021-05-27,157563,59937,delivered,credit_card,mobile,social_media


In [8]:
print(f"Any NA/NaN values: {df_orders_new.isna().any().any()}")
print(f"Any Inf values: {np.isinf(df_orders_new.select_dtypes(include=[np.number])).any().any()}")

Any NA/NaN values: False
Any Inf values: False


In [9]:
df_orders_new[df_orders_new['customer_id'] == 1]

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
4023,5280,2012-07-25,1,15201,delivered,cod,desktop,paid_search
143252,184922,2014-05-31,1,15201,returned,credit_card,mobile,referral
238890,308113,2015-07-31,1,15201,delivered,cod,mobile,paid_search
374571,483190,2017-04-23,1,15201,delivered,cod,mobile,paid_search
544446,702081,2020-02-24,1,15201,delivered,credit_card,mobile,organic_search
586950,756884,2021-04-24,1,15201,paid,credit_card,desktop,social_media


In [10]:
gaps = df_orders_new[df_orders_new['customer_id'] == 1]['order_date'].diff().dt.days.astype(float)
print(gaps.tolist()[1:]) 

[675.0, 426.0, 632.0, 1037.0, 425.0]


In [11]:
gaps = df_orders_new.groupby('customer_id')['order_date'].diff().dt.days.dropna()

In [12]:
print(np.median(gaps))

144.0


**Q1_Answer:** C) 144 ngày

# Question 2:
Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?

In [13]:
PRODUCTS = DATA_DIR + 'products.csv'
df_products = pd.read_csv(PRODUCTS)
df_products.head(10)

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406
5,541,SaigonFlex UC-06,Streetwear,Everyday,M,black,16095.853846,15291.061154
6,542,SaigonFlex UC-07,Streetwear,Everyday,L,orange,9213.817742,6152.066106
7,543,SaigonFlex UC-08,Streetwear,Everyday,XL,blue,40.394178,22.487748
8,544,SaigonFlex UC-09,Streetwear,Everyday,S,white,11334.106452,8528.915105
9,545,SaigonFlex UC-10,Streetwear,Everyday,M,purple,10045.678411,9091.338962


In [14]:
print(f"Any NA/NaN values: {df_products.isna().any().any()}")
print(f"Any Inf values: {np.isinf(df_products.select_dtypes(include=[np.number])).any().any()}")

Any NA/NaN values: False
Any Inf values: False


In [15]:
df_products_groupid = df_products.groupby(['segment']).size().reset_index(name='count')
df_products_groupid

,segment,count
0,Activewear,598
1,All-weather,169
2,Balanced,306
3,Everyday,405
4,Performance,347
5,Premium,177
6,Standard,262
7,Trendy,148


In [16]:
result = df_products.groupby('segment')[['price', 'cogs']].sum()
result

,price,cogs
segment,,
Activewear,1.553661e+06,1.216784e+06
All-weather,6.531421e+05,5.084402e+05
Balanced,2.824454e+06,2.212628e+06
Everyday,3.057421e+06,2.433037e+06
Performance,2.280777e+06,1.768000e+06
Premium,4.226185e+05,3.413006e+05
Standard,7.672912e+05,5.919552e+05
Trendy,3.274922e+05,2.583084e+05


In [17]:
df_products['margin_pct'] = (df_products['price'] - df_products['cogs']) / df_products['price']
avg_margin_by_segment = df_products.groupby('segment')['margin_pct'].mean()
print(avg_margin_by_segment)

segment
Activewear     0.265600
All-weather    0.284176
Balanced       0.258038
Everyday       0.236343
Performance    0.263650
Premium        0.285377
Standard       0.313442
Trendy         0.240758
Name: margin_pct, dtype: float64


**Q2_Answer:** D) Standard

# Question 3:
Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [18]:
RETURNS = DATA_DIR +'returns.csv'
df_returns = pd.read_csv(RETURNS)
df_returns.head(10)

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76
5,RET-000006,59,671,2012-07-19,defective,1,10086.33
6,RET-000007,67,604,2012-07-16,wrong_size,1,5713.22
7,RET-000008,102,467,2012-07-17,defective,1,9724.09
8,RET-000010,108,635,2012-07-30,wrong_size,5,43387.54
9,RET-000012,132,103,2012-07-29,changed_mind,2,19200.35


In [19]:
returns_products_merge = pd.merge(df_returns, df_products, on='product_id')
returns_products_merge.head(10)

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount,product_name,category,segment,size,color,price,cogs,margin_pct
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01,SaigonFlex UC-74,Streetwear,Everyday,M,yellow,10426.571034,8987.704231,0.1380
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37,SaigonCore YY-57,GenZ,Trendy,L,orange,2656.232069,1842.628186,0.3063
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95,VietMotion UC-07,Streetwear,Everyday,XL,yellow,5399.825901,3136.758866,0.4191
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75,VietMode RP-41,Outdoor,Activewear,M,yellow,1802.115000,1575.769356,0.1256
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76,VietMode RP-42,Outdoor,Activewear,L,red,1802.115000,1309.056336,0.2736
5,RET-000006,59,671,2012-07-19,defective,1,10086.33,SaigonFlex UC-36,Streetwear,Everyday,XL,black,11194.626316,7866.463912,0.2973
6,RET-000007,67,604,2012-07-16,wrong_size,1,5713.22,SaigonFlex UC-69,Streetwear,Everyday,S,white,6135.298831,4141.940241,0.3249
7,RET-000008,102,467,2012-07-17,defective,1,9724.09,SaigonFlex UM-72,Streetwear,Balanced,XL,silver,11081.167629,6350.617168,0.4269
8,RET-000010,108,635,2012-07-30,wrong_size,5,43387.54,SaigonFlex UC-00,Streetwear,Everyday,XL,purple,10745.220588,9205.430478,0.1433
9,RET-000012,132,103,2012-07-29,changed_mind,2,19200.35,DragonWear UM-30,Streetwear,Balanced,XL,blue,10425.177091,7115.183365,0.3175


In [20]:
filtered_streetware = returns_products_merge[returns_products_merge['category'] == 'Streetwear']
filtered_streetware = filtered_streetware.groupby(['return_reason']).size().reset_index(name='count')
filtered_streetware

,return_reason,count
0,changed_mind,3830
1,defective,4330
2,late_delivery,2159
3,not_as_described,3854
4,wrong_size,7626


**Q3_Answer:** B) wrong_size

# Question 4:
Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [21]:
TRAFFIC = DATA_DIR + 'web_traffic.csv'
df_traffic = pd.read_csv(TRAFFIC)
df_traffic.head(10)

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral
5,2013-01-06,9545,6992,47160,0.00438,263.6,social_media
6,2013-01-07,10203,7354,32749,0.00443,252.1,organic_search
7,2013-01-08,9456,7459,31482,0.00440,192.4,paid_search
8,2013-01-09,9162,7108,46717,0.00409,312.8,organic_search
9,2013-01-10,9887,7878,38472,0.00544,285.4,organic_search


In [22]:
print(f"Any NA/NaN values: {df_traffic.isna().any().any()}")
print(f"Any Inf values: {np.isinf(df_traffic.select_dtypes(include=[np.number])).any().any()}")

Any NA/NaN values: False
Any Inf values: False


In [23]:
df_traffic['bounces'] = df_traffic['sessions'] * df_traffic['bounce_rate']
traffic_agg = df_traffic.groupby('traffic_source')[['sessions', 'bounces']].sum().reset_index()
traffic_agg['true_bounce_rate'] = traffic_agg['bounces'] / traffic_agg['sessions']
traffic_agg.sort_values(by='true_bounce_rate', ascending=True)

,traffic_source,sessions,bounces,true_bounce_rate
1,email_campaign,12792670,56738.72409,0.004435
5,social_media,15816226,70773.83948,0.004475
3,paid_search,19598271,87787.19618,0.004479
2,organic_search,27196976,122345.17197,0.004498
0,direct,6571549,29709.88723,0.004521
4,referral,9476845,42899.95017,0.004527


**Q4_Answer:** C) email_campaign

# Question 5:
Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

In [24]:
ITEMS = DATA_DIR + 'order_items.csv'
df_items = pd.read_csv(ITEMS)
df_items.head(10)

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN
5,7,1934,6,1633.49,0.0,NaN,NaN
6,8,1934,6,1602.92,0.0,NaN,NaN
7,8,1935,4,1642.51,0.0,NaN,NaN
8,9,1432,8,4049.64,0.0,NaN,NaN
9,10,1431,5,3977.37,0.0,NaN,NaN


In [25]:
non_null_count = df_items['promo_id'].count()
total_count = len(df_items)
percentage_non_null = (non_null_count / total_count) * 100
print(f"Non-null percentage: {percentage_non_null:.2f}%")

Non-null percentage: 38.66%


**Q5_Answer:** C) 39%

# Question 6:
Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [26]:
CUSTOMERS = DATA_DIR + 'customers.csv'
df_customers = pd.read_csv(CUSTOMERS)
df_customers.head(10)

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search
5,6,15202,Phu Ly,2022-04-14,Female,25-34,organic_search
6,8,15202,Phu Ly,2015-09-11,Male,45-54,social_media
7,9,15202,Phu Ly,2020-02-14,Male,35-44,email_campaign
8,10,15202,Phu Ly,2014-03-03,Male,25-34,organic_search
9,11,15203,Viet Tri,2017-11-07,Male,35-44,organic_search


In [27]:
print(f"Any NA/NaN values: {df_customers.isna().any().any()}")
print(f"Any Inf values: {np.isinf(df_customers.select_dtypes(include=[np.number])).any().any()}")

Any NA/NaN values: False
Any Inf values: False


In [28]:
customers_orders_merge = pd.merge(df_customers, df_orders, on='customer_id')
customers_orders_merge.head(10)

,customer_id,zip_x,city,signup_date,gender,age_group,acquisition_channel,order_id,order_date,zip_y,order_status,payment_method,device_type,order_source
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,5280,2012-07-25,15201,delivered,cod,desktop,paid_search
1,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,184922,2014-05-31,15201,returned,credit_card,mobile,referral
2,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,308113,2015-07-31,15201,delivered,cod,mobile,paid_search
3,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,483190,2017-04-23,15201,delivered,cod,mobile,paid_search
4,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,702081,2020-02-24,15201,delivered,credit_card,mobile,organic_search
5,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,756884,2021-04-24,15201,paid,credit_card,desktop,social_media
6,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign,117673,2013-09-20,15201,delivered,paypal,mobile,social_media
7,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign,125576,2013-10-28,15201,cancelled,credit_card,desktop,direct
8,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign,231562,2014-11-13,15201,cancelled,bank_transfer,desktop,organic_search
9,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign,815219,2022-07-06,15201,delivered,apple_pay,mobile,paid_search


In [29]:
temp = customers_orders_merge.groupby(['age_group']).size().reset_index(name='count_order')
temp

,age_group,count_order
0,18-24,89057
1,25-34,190622
2,35-44,170368
3,45-54,124138
4,55+,72760


In [30]:
total_per_age = df_customers.groupby(['age_group']).size().reset_index(name='count_num')
total_per_age

,age_group,count_num
0,18-24,17039
1,25-34,36342
2,35-44,31920
3,45-54,23172
4,55+,13457


In [31]:
merged2 = pd.merge(total_per_age, temp, on='age_group')
merged2['avg_orders_per_person'] = merged2['count_order'] / merged2['count_num']
merged2.sort_values(by='avg_orders_per_person', ascending=False)

,age_group,count_num,count_order,avg_orders_per_person
4,55+,13457,72760,5.406851
3,45-54,23172,124138,5.357241
2,35-44,31920,170368,5.337343
1,25-34,36342,190622,5.245226
0,18-24,17039,89057,5.226656


**Q6_Answer:** A) 55+

# Question 7:
Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

In [35]:
PAYMENTS = DATA_DIR + 'payments.csv'
GEOGRAPHY = DATA_DIR + 'geography.csv'
df_payments = pd.read_csv(PAYMENTS)
df_geography = pd.read_csv(GEOGRAPHY)
df_geography.head()

,zip,city,region,district
0,15201,Hai Phong,East,District #13
1,15202,Phu Ly,East,District #13
2,15203,Viet Tri,East,District #13
3,15204,Bac Giang,East,District #13
4,15205,Bac Giang,East,District #13


In [36]:
df_payments.head()

,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1


In [38]:
pay_ord = pd.merge(df_payments, df_orders, on='order_id', how='inner')
pay_ord.head(10)

,order_id,payment_method_x,payment_value,installments,order_date,customer_id,zip,order_status,payment_method_y,device_type,order_source
0,1,credit_card,7967.54,3,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,cod,71163.75,1,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,credit_card,33660.99,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,credit_card,53196.25,3,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,paypal,1597.84,1,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign
5,7,credit_card,9800.94,12,2012-07-06,57820,2886,delivered,credit_card,tablet,organic_search
6,8,credit_card,16187.56,6,2012-07-06,57818,2886,delivered,credit_card,mobile,email_campaign
7,9,apple_pay,32397.12,12,2012-07-06,49102,5262,delivered,apple_pay,tablet,paid_search
8,10,paypal,19886.85,1,2012-07-06,49101,5262,delivered,paypal,desktop,organic_search
9,13,credit_card,36037.68,3,2012-07-06,40638,7456,delivered,credit_card,tablet,social_media


In [39]:
pay_ord_cust = pd.merge(pay_ord, df_customers, on='customer_id', how='inner')
pay_ord_cust.head(10)

,order_id,payment_method_x,payment_value,installments,order_date,customer_id,zip_x,order_status,payment_method_y,device_type,order_source,zip_y,city,signup_date,gender,age_group,acquisition_channel
0,1,credit_card,7967.54,3,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,1109,Hanoi,2020-06-06,Female,35-44,social_media
1,2,cod,71163.75,1,2012-07-04,58621,1330,returned,cod,mobile,paid_search,1330,Phu Ly,2021-11-03,Female,18-24,social_media
2,3,credit_card,33660.99,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,1473,Lao Cai,2020-09-18,Female,35-44,direct
3,4,credit_card,53196.25,3,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,2360,Son Tay,2016-05-29,Male,45-54,direct
4,6,paypal,1597.84,1,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,2886,Uong Bi,2017-07-11,Male,18-24,social_media
5,7,credit_card,9800.94,12,2012-07-06,57820,2886,delivered,credit_card,tablet,organic_search,2886,Uong Bi,2014-09-22,Female,45-54,paid_search
6,8,credit_card,16187.56,6,2012-07-06,57818,2886,delivered,credit_card,mobile,email_campaign,2886,Uong Bi,2017-04-26,Female,35-44,paid_search
7,9,apple_pay,32397.12,12,2012-07-06,49102,5262,delivered,apple_pay,tablet,paid_search,5262,Thai Nguyen,2016-01-02,Female,18-24,social_media
8,10,paypal,19886.85,1,2012-07-06,49101,5262,delivered,paypal,desktop,organic_search,5262,Thai Nguyen,2022-01-22,Female,45-54,email_campaign
9,13,credit_card,36037.68,3,2012-07-06,40638,7456,delivered,credit_card,tablet,social_media,7456,Bac Ninh,2017-09-10,Male,45-54,email_campaign


In [41]:
full_data = pd.merge(pay_ord_cust, df_geography, left_on='zip_x', right_on='zip', how='inner')
full_data.head(10)

,order_id,payment_method_x,payment_value,installments,order_date,customer_id,zip_x,order_status,payment_method_y,device_type,...,zip_y,city_x,signup_date,gender,age_group,acquisition_channel,zip,city_y,region,district
0,1,credit_card,7967.54,3,2012-07-04,58578,1109,delivered,credit_card,desktop,...,1109,Hanoi,2020-06-06,Female,35-44,social_media,1109,Hanoi,East,District #02
1,2,cod,71163.75,1,2012-07-04,58621,1330,returned,cod,mobile,...,1330,Phu Ly,2021-11-03,Female,18-24,social_media,1330,Phu Ly,East,District #02
2,3,credit_card,33660.99,3,2012-07-04,58811,1473,delivered,credit_card,desktop,...,1473,Lao Cai,2020-09-18,Female,35-44,direct,1473,Lao Cai,East,District #02
3,4,credit_card,53196.25,3,2012-07-04,59453,2360,delivered,credit_card,desktop,...,2360,Son Tay,2016-05-29,Male,45-54,direct,2360,Son Tay,East,District #02
4,6,paypal,1597.84,1,2012-07-06,57821,2886,delivered,paypal,mobile,...,2886,Uong Bi,2017-07-11,Male,18-24,social_media,2886,Uong Bi,East,District #02
5,7,credit_card,9800.94,12,2012-07-06,57820,2886,delivered,credit_card,tablet,...,2886,Uong Bi,2014-09-22,Female,45-54,paid_search,2886,Uong Bi,East,District #02
6,8,credit_card,16187.56,6,2012-07-06,57818,2886,delivered,credit_card,mobile,...,2886,Uong Bi,2017-04-26,Female,35-44,paid_search,2886,Uong Bi,East,District #02
7,9,apple_pay,32397.12,12,2012-07-06,49102,5262,delivered,apple_pay,tablet,...,5262,Thai Nguyen,2016-01-02,Female,18-24,social_media,5262,Thai Nguyen,East,District #01
8,10,paypal,19886.85,1,2012-07-06,49101,5262,delivered,paypal,desktop,...,5262,Thai Nguyen,2022-01-22,Female,45-54,email_campaign,5262,Thai Nguyen,East,District #01
9,13,credit_card,36037.68,3,2012-07-06,40638,7456,delivered,credit_card,tablet,...,7456,Bac Ninh,2017-09-10,Male,45-54,email_campaign,7456,Bac Ninh,East,District #03


In [42]:
revenue_by_region = full_data.groupby('region')['payment_value'].sum().reset_index()
revenue_by_region.sort_values(by='payment_value', ascending=False)

,region,payment_value
1,East,7.291151e+09
0,Central,4.719491e+09
2,West,3.670227e+09


**Q7_Answer:** C) East

# Question 8:
Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

In [44]:
cancelled = df_orders[df_orders['order_status'] == 'cancelled'].groupby('payment_method').size().reset_index(name='count')
cancelled.sort_values(by='count', ascending=False)

,payment_method,count
3,credit_card,28452
2,cod,15468
4,paypal,7817
0,apple_pay,5190
1,bank_transfer,2535


**Q8_Answer:** A) credit_card

# Question 9:
Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)? 

In [45]:
ITEMS = DATA_DIR + 'order_items.csv'
df_items = pd.read_csv(ITEMS)
items_prod = pd.merge(df_items, df_products, on='product_id', how='inner')
items_prod.head(10)

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,product_name,category,segment,size,color,price,cogs,margin_pct
0,1,2400,7,1138.22,0.0,NaN,NaN,VietMotion YY-09,GenZ,Trendy,S,red,1109.261061,1053.798008,0.0500
1,2,609,7,10166.25,0.0,NaN,NaN,SaigonFlex UC-74,Streetwear,Everyday,M,yellow,10426.571034,8987.704231,0.1380
2,3,396,3,11220.33,0.0,NaN,NaN,SaigonFlex UM-01,Streetwear,Balanced,S,green,11028.428695,10091.012256,0.0850
3,4,635,5,10639.25,0.0,NaN,NaN,SaigonFlex UC-00,Streetwear,Everyday,XL,purple,10745.220588,9205.430478,0.1433
4,6,1935,1,1597.84,0.0,NaN,NaN,UrbanVN RP-10,Outdoor,Activewear,XL,purple,1609.911509,1048.696357,0.3486
5,7,1934,6,1633.49,0.0,NaN,NaN,UrbanVN RP-09,Outdoor,Activewear,L,white,1609.911509,1337.836464,0.1690
6,8,1934,6,1602.92,0.0,NaN,NaN,UrbanVN RP-09,Outdoor,Activewear,L,white,1609.911509,1337.836464,0.1690
7,8,1935,4,1642.51,0.0,NaN,NaN,UrbanVN RP-10,Outdoor,Activewear,XL,purple,1609.911509,1048.696357,0.3486
8,9,1432,8,4049.64,0.0,NaN,NaN,VietMode RP-24,Outdoor,Activewear,S,orange,4093.740000,3889.053000,0.0500
9,10,1431,5,3977.37,0.0,NaN,NaN,VietMode RP-23,Outdoor,Activewear,XL,black,4093.740000,2673.212220,0.3470


In [59]:
total_sold_by_size = items_prod.groupby('size').size().reset_index(name='total_sold')
total_sold_by_size

,size,total_sold
0,L,173174
1,M,176428
2,S,172042
3,XL,193025


In [49]:
RETURNS = DATA_DIR + 'returns.csv'
df_returns = pd.read_csv(RETURNS)
returns_prod = pd.merge(df_returns, df_products, on='product_id', how='inner')
returns_prod.head(10)

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount,product_name,category,segment,size,color,price,cogs,margin_pct
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01,SaigonFlex UC-74,Streetwear,Everyday,M,yellow,10426.571034,8987.704231,0.1380
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37,SaigonCore YY-57,GenZ,Trendy,L,orange,2656.232069,1842.628186,0.3063
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95,VietMotion UC-07,Streetwear,Everyday,XL,yellow,5399.825901,3136.758866,0.4191
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75,VietMode RP-41,Outdoor,Activewear,M,yellow,1802.115000,1575.769356,0.1256
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76,VietMode RP-42,Outdoor,Activewear,L,red,1802.115000,1309.056336,0.2736
5,RET-000006,59,671,2012-07-19,defective,1,10086.33,SaigonFlex UC-36,Streetwear,Everyday,XL,black,11194.626316,7866.463912,0.2973
6,RET-000007,67,604,2012-07-16,wrong_size,1,5713.22,SaigonFlex UC-69,Streetwear,Everyday,S,white,6135.298831,4141.940241,0.3249
7,RET-000008,102,467,2012-07-17,defective,1,9724.09,SaigonFlex UM-72,Streetwear,Balanced,XL,silver,11081.167629,6350.617168,0.4269
8,RET-000010,108,635,2012-07-30,wrong_size,5,43387.54,SaigonFlex UC-00,Streetwear,Everyday,XL,purple,10745.220588,9205.430478,0.1433
9,RET-000012,132,103,2012-07-29,changed_mind,2,19200.35,DragonWear UM-30,Streetwear,Balanced,XL,blue,10425.177091,7115.183365,0.3175


In [58]:
total_returned_by_size = returns_prod.groupby('size').size().reset_index(name='total_returned')
total_returned_by_size

,size,total_returned
0,L,9741
1,M,9820
2,S,9723
3,XL,10655


In [52]:
final_q9 = pd.merge(total_sold_by_size, total_returned_by_size, on='size', how='inner')
final_q9['return_rate'] = final_q9['total_returned'] / final_q9['total_sold']
final_q9.sort_values(by='return_rate', ascending=False)

,size,total_sold,total_returned,return_rate
2,S,172042,9723,0.056515
0,L,173174,9741,0.056250
1,M,176428,9820,0.055660
3,XL,193025,10655,0.055200


**Q9_Answer:** A) S

# Question 10:
Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

In [53]:
PAYMENTS = DATA_DIR + 'payments.csv'
df_payments = pd.read_csv(PAYMENTS)
df_payments.head(10)

,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1
5,7,credit_card,9800.94,12
6,8,credit_card,16187.56,6
7,9,apple_pay,32397.12,12
8,10,paypal,19886.85,1
9,13,credit_card,36037.68,3


In [54]:
total_install = df_payments.groupby(['installments'])['payment_value'].sum().reset_index(name='sum')
total_install

,installments,sum
0,1,6.338560e+09
1,2,7.750703e+05
2,3,5.342276e+09
3,6,2.686932e+09
4,12,1.312327e+09


In [55]:
num_install = df_payments.groupby(['installments']).size().reset_index(name='num')
num_install

,installments,num
0,1,262866
1,2,1094
2,3,218949
3,6,109910
4,12,54126


In [57]:
avg = pd.merge(total_install, num_install, on='installments')
avg['avg_payment_value'] = avg['sum'] / avg['num']
avg.sort_values(by='avg_payment_value', ascending=False)

,installments,sum,num,avg_payment_value
3,6,2.686932e+09,109910,24446.654403
2,3,5.342276e+09,218949,24399.635486
4,12,1.312327e+09,54126,24245.772694
0,1,6.338560e+09,262866,24113.274166
1,2,7.750703e+05,1094,708.473729


**Q10_Answer:** C) 6 kỳ